# 🎯 Objetivo del notebook

Crear nuevas variables (“features”) que potencien la detección de anomalías, inspirándonos en:

- Técnicas automatizadas (usando librerías como featuretools y scikit-learn).
- Criterios derivados del paper:

Gradientes/derivadas de la señal CCL (identifican cambios bruscos).
Ventanas móviles (rolling stats para evaluar contexto local).
Normalización por máximo CCL, como sugiere el paper.
Agrupación por pozo, etapa y sentido para trabajar segmentadamente.
Posible creación de una señal “anómala” cuando CCL se desvía de lo esperado.

## 🧪 Feature Engineering para Detección de Anomalías CCL

Este notebook genera nuevas variables a partir de los datos originales, con el fin de alimentar un modelo de detección de anomalías.

Nos basamos en dos enfoques:

1. **Guía técnica del paper** sobre el Anomaly Detector Tool (ADT).
2. **Técnicas automatizadas de ingeniería de features**.

Se generan características como derivadas, estadísticas móviles, normalización personalizada y estadísticas agregadas por pozo y etapa.


## 📘 Celda: Filtros interactivos por pozo, etapa y sentido

In [1]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Cargar datos
df_raw = pd.read_csv(r"C:\Developer\fundamentos\data\salida_DEPT_CCL_TENS_FILTRADA.csv")

# Dropdowns interactivos
pozo_dropdown = widgets.Dropdown(
    options=sorted(df_raw["pozo"].dropna().unique()),
    description="Pozo:"
)

sentido_dropdown = widgets.Dropdown(
    options=sorted(df_raw["sentido"].dropna().unique()),
    description="Sentido:"
)

# Mostrar filtros
display(pozo_dropdown, sentido_dropdown)


Dropdown(description='Pozo:', options=('BPE-2341', 'BPE-2342', 'BPE-2343', 'BPO-2702', 'BPO-2703'), value='BPE…

Dropdown(description='Sentido:', options=('Down', 'Up'), value='Down')

## 📂 Celda 2 - Carga de datos

In [12]:
# Aplicar filtros seleccionados con etapas definidas manualmente
etapas_seleccionadas = ['E35', 'E36', 'E37', 'E38', 'E46', 'E48']  # Modifica esta lista con tus etapas deseadas

df = df_raw[
    (df_raw["pozo"] == pozo_dropdown.value) &
    (df_raw["etapa"].isin(etapas_seleccionadas)) &  # Filtro con lista manual de etapas
    (df_raw["sentido"] == sentido_dropdown.value) &
    (df_raw["CCL"] > -0.05) &
    (df_raw["CCL"] < 0.05)
].copy()
df.reset_index(drop=True, inplace=True)

print(f"Filas seleccionadas: {len(df)}")

# Vista general del dataset completo
print(f"Total de filas: {len(df)}")
print("Pozos disponibles:", df["pozo"].unique())
print("Sentidos disponibles:", df["sentido"].unique())

# Vista rápida
df.head()

Filas seleccionadas: 97242
Total de filas: 97242
Pozos disponibles: ['BPE-2343']
Sentidos disponibles: ['Up']


,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa
0,2800.05,-0.00004,1732.00004,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48
1,2800.10,-0.00212,1732.00004,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48
2,2800.15,0.00143,1669.99998,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48
3,2800.20,0.00558,1669.99998,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48
4,2800.25,0.00775,1669.99998,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48


### 🧮 Celda 3 - Normalización del CCL como en el paper

#### ⚙️ Normalización del CCL (según paper)

El paper menciona que CCL debe normalizarse multiplicando por 10 y dividiendo por su valor máximo dentro del run.


In [13]:
# Normalizamos CCL por pozo-etapa-sentido como contexto lógico
df["CCL_norm"] = df.groupby(["pozo", "etapa", "sentido"])["CCL"].transform(
    lambda x: (x * 10) / x.max()
)

### 🧾 Celda 4 - Derivadas y gradientes de CCL

#### 📉 Gradiente de la señal CCL

Calculamos la diferencia entre puntos consecutivos de CCL normalizado, como indicador de variaciones abruptas.


In [14]:
df["dCCL"] = df.groupby(["pozo", "etapa", "sentido"])["CCL_norm"].diff()
df["abs_dCCL"] = df["dCCL"].abs()


### 🪟 Celda 5 - Estadísticas de ventana móvil

#### 📊 Estadísticas móviles (rolling)

Promedios, máximos, mínimos y desviaciones estándar en una ventana móvil de 5 y 10 muestras, para capturar el “contexto local” de la señal.


In [15]:
for window in [5, 10]:
    df[f"CCL_roll_mean_{window}"] = df.groupby(["pozo", "etapa", "sentido"])["CCL_norm"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    df[f"CCL_roll_std_{window}"] = df.groupby(["pozo", "etapa", "sentido"])["CCL_norm"].transform(lambda x: x.rolling(window=window, min_periods=1).std())
    df[f"CCL_roll_max_{window}"] = df.groupby(["pozo", "etapa", "sentido"])["CCL_norm"].transform(lambda x: x.rolling(window=window, min_periods=1).max())
    df[f"CCL_roll_min_{window}"] = df.groupby(["pozo", "etapa", "sentido"])["CCL_norm"].transform(lambda x: x.rolling(window=window, min_periods=1).min())


### 🧮 Celda 6 - Ratio entre señal y su contexto

#### 🔍 Ratios entre CCL actual y media local

Calculamos la razón entre el CCL actual y el promedio/mínimo/máximo locales como indicadores de posibles outliers.


In [16]:
df["ratio_to_mean5"] = df["CCL_norm"] / (df["CCL_roll_mean_5"] + 1e-5)
df["ratio_to_std5"] = df["abs_dCCL"] / (df["CCL_roll_std_5"] + 1e-5)

### 🔬 Celda 7 - Estadísticas agregadas por etapa

#### 🧱 Estadísticas por etapa

Creamos features agregadas para cada combinación pozo-etapa-sentido.

In [17]:
agg_features = df.groupby(["pozo", "etapa", "sentido"]).agg({
    "CCL_norm": ["mean", "std", "max", "min"],
    "abs_dCCL": ["mean", "std", "max"],
    "TENS": ["mean", "std", "max"],
})

# Renombramos columnas
agg_features.columns = ['_'.join(col).strip() for col in agg_features.columns.values]
agg_features.reset_index(inplace=True)

agg_features.head()


,pozo,etapa,sentido,CCL_norm_mean,CCL_norm_std,CCL_norm_max,CCL_norm_min,abs_dCCL_mean,abs_dCCL_std,abs_dCCL_max,TENS_mean,TENS_std,TENS_max
0,BPE-2343,E36,Up,0.138756,3.094922,10.0,-9.997998,1.296902,1.527585,19.815816,2301.805072,252.059941,2816.99993
1,BPE-2343,E38,Up,0.164780,1.987944,10.0,-10.014031,0.959726,1.390248,18.757266,1670.112075,125.650718,2115.00006
2,BPE-2343,E46,Up,0.124819,3.217592,10.0,-9.985997,1.315344,1.445609,19.795959,2028.417809,201.573888,2359.99932
3,BPE-2343,E48,Up,0.078532,1.799373,10.0,-9.991994,0.797665,1.187111,19.481585,1583.567427,158.066178,2023.00015


### 🧪 🔧 Celda 8: Código para detectar y marcar cuplas

Este bloque de código nos permite:

- Calcula la derivada de la señal CCL (dCCL) y su valor absoluto (abs_dCCL).
- Detecta picos bruscos (posibles cuplas) usando un umbral (ej. percentil 99).
- Verifica si aparecen cada ~15 metros.
- Marca esos puntos como es_cupla = True.
- Permite excluirlos del análisis o tratarlos aparte.

In [18]:
# df contiene la columna 'CCL' ordenada por DEPT
# df = df.sort_values(by="DEPT").copy()

# 1. Calcular derivada y módulo
# df["dCCL"] = df["CCL"].diff()
# df["abs_dCCL"] = df["dCCL"].abs()

# 2. Definir umbral para detectar picos (ajustable)
# umbral_cupla = df["abs_dCCL"].quantile(0.99)  # Top 1% de cambios
# df["es_cupla"] = df["abs_dCCL"] > umbral_cupla

# 3. Ubicación de las cuplas detectadas
# cuplas_detectadas = df[df["es_cupla"]]["DEPT"].values
# distancias = np.diff(cuplas_detectadas)

# print("📏 Distancias entre cuplas detectadas:")
# print(distancias)

# 4. Estadísticas
# print(f"Media: {np.mean(distancias):.2f} m | Std: {np.std(distancias):.2f} m")


#### Excluir las cuplas para modelado

In [19]:
# df_modelo = df[~df["es_cupla"]].copy()

### 💾 Celda 9 - Exportación para modelado

#### 💾 Exportación del dataset con features

Guardamos el dataset enriquecido para la siguiente etapa de detección de anomalías.

In [20]:
# Unimos features agregadas de vuelta al dataset base
df = df.merge(agg_features, on=["pozo", "etapa", "sentido"], how="left")

# Guardamos para próxima etapa
df.to_csv(r"C:\Developer\fundamentos\data\ccl_features.csv", index=False)